# Orbit Wars Set Transformer Modeli

Bu defter `model.py` dosyasındaki `OrbitWarsPolicy` ağını hem teorik hem de pratik açıdan açıklar. Amaç sadece kodun ne yaptığını göstermek değil; neden Set Transformer seçildiğini, attention bias ve mask mekanizmalarının oyundaki karar problemine nasıl bağlandığını da netleştirmektir.

Modelin özeti:

- Girdi, sabit uzunluklu ama maskelenmiş bir token setidir: gezegenler, filolar ve bir global token.
- Encoder, self-attention kullanan bir Set Transformer benzeridir.
- Pairwise geometri `edge_bias` ile attention skorlarına doğrudan eklenir.
- Actor, aksiyonu üç parçaya ayırır: kaynak gezegen, hedef gezegen, gönderilecek gemi oranı.
- Critic, global token üzerinden durum değerini üretir.

## 1. Problem: Orbit Wars Durumu Neden Set Gibi?

Orbit Wars gözlemi doğal olarak sabit uzunluklu bir vektör değildir. Her turda farklı sayıda gezegen ve havada farklı sayıda filo olabilir. Ayrıca gezegenlerin listedeki sırası semantik olarak önemli değildir: aynı gezegen kümesini farklı sırada vermek, oyunun stratejik anlamını değiştirmez.

Bu yüzden model şu fikre yaslanır:

$$X = \{x_1, x_2, \ldots, x_T\}$$

Burada her $x_i$ bir token temsilidir. Tokenlar üç tiptir:

| Token tipi | Kaynak | Anlam |
| --- | --- | --- |
| Planet token | `planet_feats` | Bir gezegenin sahiplik, gemi, üretim, konum ve hareket bilgisi |
| Fleet token | `fleet_feats` | Havadaki bir filonun sahiplik, gemi, hız, yön ve ETA bilgisi |
| Global token | `global_feats` | Tur, toplam güç, üretim, gelen tehdit gibi global özet |

`model.py` bu token sırasını sabit kabul eder:

```text
[planet_0, ..., planet_{P_max-1}, fleet_0, ..., fleet_{F_max-1}, GLOBAL]
```

Toplam token sayısı:

$$T = P_{max} + F_{max} + 1$$

## 2. Notebook Kurulumu

Aşağıdaki hücre, defter `orbit-wars-data/notebooks/notebooks` altında çalıştırılsa da repo kökünü import path'e ekler.

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while REPO.name != 'Orbit_Wars' and REPO.parent != REPO:
    REPO = REPO.parent

sys.path.insert(0, str(REPO))
print('Repo root:', REPO)

Repo root: /home/doruk/Documents/Kaggle/Orbit_Wars


In [2]:
import math
import torch

from obs_builder import build_observation, decode_action
from model import OrbitWarsPolicy, pack_observations

## 3. Oyuncak Bir Observation

Modeli anlatmak için küçük ve deterministik bir oyun durumu kullanacağız. Burada `p_max=8`, `f_max=4` seçiyoruz; gerçek default değerler `48` (gezegen tavanı) ve `64` (filo tavanı) — kaggle env mekaniğinden türetilmiş worst-case sınırları.

In [3]:
toy_obs = {
    'planets': [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],
        [1, 0, 80.0, 70.0, 2.0, 50, 2],
        [2, 1, 30.0, 30.0, 2.0, 80, 2],
        [3, -1, 20.0, 80.0, 2.0, 10, 1],
    ],
    'fleets': [
        [10, 1, 60.0, 60.0, 0.0, 2, 25],
    ],
    'initial_planets': [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],
        [1, 0, 80.0, 70.0, 2.0, 50, 2],
        [2, 1, 30.0, 30.0, 2.0, 80, 2],
        [3, -1, 20.0, 80.0, 2.0, 10, 1],
    ],
    'comet_planet_ids': [],
    'comets': [],
}

obs = build_observation(toy_obs, player=0, angular_velocity=0.05, p_max=8, f_max=4)

print('planet_feats:', obs.planet_feats.shape)
print('fleet_feats :', obs.fleet_feats.shape)
print('global_feats:', obs.global_feats.shape)
print('edge_bias   :', obs.edge_bias.shape)
print('planet_mask :', obs.planet_mask.astype(int))
print('fleet_mask  :', obs.fleet_mask.astype(int))
print('source_mask :', obs.source_mask.astype(int))

planet_feats: (8, 15)
fleet_feats : (4, 11)
global_feats: (12,)
edge_bias   : (13, 13, 5)
planet_mask : [1 1 1 1 0 0 0 0]
fleet_mask  : [1 0 0 0]
source_mask : [1 1 0 0 0 0 0 0]


## 4. Feature Pipeline ve Maskler

`obs_builder.py` modelin beklediği sabit şekilli tensörleri üretir. Buradaki kritik fikir padding + mask yaklaşımıdır.

| Alan | Şekil | Ne işe yarar? |
| --- | --- | --- |
| `planet_feats` | `(P_max, 23)` | Gezegen token ham özellikleri; sahiplik/gemi/üretim/konum + incoming self/enemy filo aggregate'leri |
| `fleet_feats` | `(F_max, 11)` | Filo token ham özellikleri |
| `global_feats` | `(12,)` | Tüm durumun global özeti |
| `edge_bias` | `(T, T, 5)` | Token çiftleri arası mesafe, dx, dy ve güneş etrafındaki açı farkının `sin/cos`'u |
| `planet_mask` | `(P_max,)` | Gerçek gezegen mi, padding mi? |
| `fleet_mask` | `(F_max,)` | Gerçek filo mu, padding mi? |
| `source_mask` | `(P_max,)` | Bu gezegenden filo çıkarılabilir mi? |

Planet feature vektörünün son 8 kanalı incoming filo bilgisidir:

```text
15-16: incoming_self_ships, incoming_self_eta_min
17-18: incoming_enemy_0_ships, incoming_enemy_0_eta_min
19-20: incoming_enemy_1_ships, incoming_enemy_1_eta_min
21-22: incoming_enemy_2_ships, incoming_enemy_2_eta_min
```

Enemy slot'ları current player dışındaki fleet owner id'lerinin sıralı haliyle atanır ve en fazla üç enemy owner tutulur. Bu bilgi target planet owner'a göre değil, **filonun owner'ına göre** sınıflanır.

Model iki farklı mask fikri kullanır:

- Padding mask: attention içinde padding tokenlara bakılmasını engeller.
- Action mask: invalid kaynak veya hedef aksiyonlara olasılık verilmesini engeller.

Bu ayrım önemli: her padding problemi attention problemi değildir; bazıları policy head seviyesinde çözülür.

## 5. Tokenlaştırma: Heterojen Veriyi Ortak Uzaya Taşımak

`OrbitWarsPolicy._build_tokens` üç farklı feature tipini ortak `d_model` boyutuna projekte eder:

$$z_i = W_{type(i)} x_i + e_{type(i)}$$

Kod karşılığı:

```python
planet_tok = self.planet_proj(planet_feats)
fleet_tok = self.fleet_proj(fleet_feats)
global_tok = self.global_proj(global_feats).unsqueeze(1)
x = torch.cat([planet_tok, fleet_tok, global_tok], dim=1)
x = x + self.type_embed(self.type_ids)[None]
```

Burada type embedding küçük ama önemli bir detaydır. Gezegen ve filo feature vektörleri farklı anlamlara gelir; projeksiyondan sonra ikisi de `d_model` boyutlu olduğu için modele token tipini ayrıca söylemek gerekir.

## 5.1 Derinlemesine: Projection = Embedding

Bu noktada şöyle bir soru akla gelebilir: LLM'lerde `nn.Embedding` ile bir token ID'nin latent uzaydaki vektörünü alıyoruz; bizde ise `fleet_feats` zaten sayısal bir vektör, üstüne `fleet_proj` ile bir kez daha lineer dönüşüm uyguluyoruz. Bu "embedding'in embedding'i" mi?

Cevap: **Hayır. Her iki durum aynı matematiksel operasyondur; sadece input modalitesi farklı.**

### LLM'de ne oluyor?

```python
# nn.Embedding(vocab_size=50000, d_model=512)
token_id = 7234
embedding[7234]  # -> (512,) vektör
```

Bu, aslında gizli bir lineer dönüşümdür:

$$\text{embedding}[i] = \text{one\_hot}(i) \cdot W, \quad W \in \mathbb{R}^{50000 \times 512}$$

Yani `nn.Embedding`, **one-hot vektörü d_model'e projekte eden bias'sız bir `Linear(vocab, d_model)` katmanıdır.** Tablo lookup sadece bu işlemi hızlıca yapmanın optimize halidir.

### Bizde ne oluyor?

```python
# nn.Linear(23, d_model)
planet_feat = [owner flags, ships, production, polar position, type flags,
               capture/log ships, incoming_self, incoming_enemy_0/1/2]
planet_proj(planet_feat)  # -> (d_model,) vektör

# nn.Linear(11, d_model)
fleet_feat = [is_self, is_enemy, ships/100, log_ships, speed, cos(a), sin(a), x, y, eta, has_pred]
fleet_proj(fleet_feat)  # -> (d_model,) vektör
```

Matematiksel olarak:

$$\text{planet\_proj}(x) = x \cdot W + b, \quad W \in \mathbb{R}^{23 \times d_{model}}$$

$$\text{fleet\_proj}(x) = x \cdot W + b, \quad W \in \mathbb{R}^{11 \times d_{model}}$$

### Karşılaştırma

| | LLM (sparse input) | Bizde (dense input) |
| --- | --- | --- |
| Input | `one_hot(token_id)` — 50000-d, %99.998 sıfır | `planet_feat` — 23-d, `fleet_feat` — 11-d, dense |
| Operation | `one_hot @ W` (= W'nin ilgili satırı) | `feat @ W + b` |
| W shape | `(vocab, d_model)` | `(23, d_model)` veya `(11, d_model)` |
| Generalization | Yok: her token bağımsız satır | Var: komşu feature değerleri komşu vektörlere düşer |
| Anlam | "Token kimliği latent uzayda nereye düşer?" | "Dense feature latent uzayda nereye düşer?" |

### Neden hâlâ gerekli?

1. **Boyut eşleme.** Transformer `d_model=128` bekliyor; planet input 23-d, fleet input 11-d. Direkt besleyemeyiz.
2. **Heterojen token tipleri.** Planet (23-d), fleet (11-d), global (12-d) farklı feature uzayları. Her biri için ayrı `*_proj` lazım; shared lookup table bunu yapamaz (shape uyuşmaz).
3. **Feature kombinasyonu öğrenmesi.** `Linear` her output dim'i tüm input dim'lerin bir kombinasyonu yapar. "Yüksek `ships` AND `incoming_enemy_0_ships`" gibi etkileşimleri ilk adımda yakalar.
4. **Type embedding eklenmesi mantıklı kalsın.** Projection sonrası `+ type_embed[planet/fleet/global]` ekliyoruz; klasik LLM'deki `+ position_embed` ile aynı pattern.

### Tek cümlelik özet

> "Embedding" demek "discrete ID → vektör" değil, "input → latent uzay" demektir. LLM'de input sparse one-hot, bizde dense feature vektörü. **Operasyon birebir aynı**: `input @ W + b`.

In [4]:
batch = pack_observations([obs])
net = OrbitWarsPolicy(p_max=8, f_max=4, d_model=32, n_heads=2, n_layers=2)

with torch.no_grad():
    x0 = net._build_tokens(batch['planet_feats'], batch['fleet_feats'], batch['global_feats'])
    attn_bias = net._build_attn_bias(batch['edge_bias'])
    padding_mask = net._build_padding_mask(batch['planet_mask'], batch['fleet_mask'])

print('tokens      :', tuple(x0.shape), '=(B, T, d_model)')
print('attn_bias  :', tuple(attn_bias.shape), '=(B, heads, T, T)')
print('padding    :', tuple(padding_mask.shape), 'True = PAD')
print('type_ids   :', net.type_ids.tolist())

tokens      : (1, 13, 32) =(B, T, d_model)
attn_bias  : (1, 2, 13, 13) =(B, heads, T, T)
padding    : (1, 13) True = PAD
type_ids   : [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2]


## 6. Self-Attention Teorisi

Standart multi-head self-attention şu üç projeksiyonu üretir:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Bir tokenın diğer tokenlara ne kadar bakacağını attention skorları belirler:

$$S = \frac{QK^T}{\sqrt{d_h}}$$

Sonra softmax ile ağırlıklar elde edilir:

$$A = \mathrm{softmax}(S)$$

Çıkış:

$$Y = AV$$

Orbit Wars açısından bu şu demektir: bir gezegen tokenı, kendi durumunu güncellerken diğer gezegenlere, düşman filolara ve global duruma bakabilir. Örneğin kaynak gezegen güçlü olsa bile yakındaki düşman filosu veya daha değerli hedef başka bir token olarak attention içinde hesaba katılabilir.

## 7. Edge Bias: Geometriyi Attention'a Katmak

`model.py` standart attention'a bir ekleme yapar:

$$S_{hij} = \frac{q_{hi} \cdot k_{hj}}{\sqrt{d_h}} + b_{hij}$$

`b_{hij}` değeri `edge_bias[i, j]` feature'larından öğrenilir. Bu vektör artık **5 boyutlu**:

```python
edge_bias[i, j] = (
    dist(i, j) / 100,
    (x_j - x_i) / 100,
    (y_j - y_i) / 100,
    sin(phi_j - phi_i),    # güneşe göre açı farkı
    cos(phi_j - phi_i),    # güneşe göre açı farkı
)
```

İlk 3 kanal kartezyen geometriyi (mesafe + işaretli yön) verir. Son 2 kanal **güneş merkezli polar açı farkını** taşır: her token için `phi = atan2(y - sun_y, x - sun_x)`. İki gezegen aynı `angular_velocity` ile döndüğü için aralarındaki `Δφ` **zamanla değişmez** — model bu sayede board döndükçe her turn yeniden öğrenmek zorunda kalmadan, kalıcı orbital ilişkiyi yakalayabilir.

```python
b = self.edge_proj(edge_bias)          # (B, T, T, H)
b = b.permute(0, 3, 1, 2)              # (B, H, T, T)
scores = scores + attn_bias
```

Bunun pratik etkisi: model mesafe, yön, yakınlık ve orbital faz ilişkisini sıfırdan koordinatlardan çıkarmak zorunda kalmaz. Attention mekanizmasına doğrudan geometrik + orbital önbilgi verilir.

Bu bir GNN edge feature kullanımına benzer; ama mesajlaşma katmanını ayrı bir graph library ile kurmak yerine Transformer attention skorlarına bias olarak eklenir.

In [5]:
with torch.no_grad():
    # İlk batch, ilk head için token 0'ın tokenlara olan learned bias değerleri.
    head0_from_planet0 = attn_bias[0, 0, 0]

print('planet_0 -> tüm tokenlar için head-0 bias:')
print(head0_from_planet0.round(decimals=3))

print('\nHam edge feature örnekleri (5 kanal: dist, dx, dy, sin(dphi), cos(dphi)):')
print('planet_0 -> planet_1:', batch['edge_bias'][0, 0, 1].tolist())
print('planet_0 -> global  :', batch['edge_bias'][0, 0, -1].tolist())

planet_0 -> tüm tokenlar için head-0 bias:
tensor([ 0.2370,  0.5820, -0.1870,  0.3600, -0.1900, -0.1900, -0.1900, -0.1900,
         0.6580, -0.1900, -0.1900, -0.1900,  0.3380])

Ham edge feature örnekleri (5 kanal: dist, dx, dy, sin(dphi), cos(dphi)):
planet_0 -> planet_1: [0.10000000149011612, 0.10000000149011612, 0.0, -0.1961161345243454, 0.9805806875228882]
planet_0 -> global  : [0.2828427255153656, -0.20000000298023224, -0.20000000298023224, -0.7071067690849304, 0.7071068286895752]


## 7.1 Derinlemesine: `edge_bias` ve `edge_proj`

Edge bias tasarımı modelin en kritik uyumlarından biri. Daha derine inelim.

### Problem: Vanilla attention token çiftini nasıl ilişkilendirir?

Standart self-attention'da skor formülü:

$$S_{ij} = \frac{Q_i \cdot K_j}{\sqrt{d}}$$

Bu skor sadece **token içeriğinden** gelir. Ama bizim tokenlarımız uzayda *konumlu* — gezegen 0 ile gezegen 1 arasındaki mesafe stratejik olarak kritik (filo 30 turn'de mi yoksa 5 turn'de mi varır?). Sadece feature'lar Q/K'ya dönüştürülünce model mesafeyi **içerikten çıkarsamak** zorunda kalır, ki bu zordur:

$$\|x_i - x_j\|^2 = \|x_i\|^2 + \|x_j\|^2 - 2 x_i \cdot x_j$$

Non-lineer bir yapı, bilineer formdan (QK) sıfırdan öğrenilmek zorunda.

### Klasik alternatifler ve neden onları seçmedik

| Yöntem | Nasıl çalışır | Bizim için neden uygun değil |
| --- | --- | --- |
| **Sinusoidal positional encoding** (Vaswani 2017) | Sıralı pozisyon → sin/cos | Gezegen sırası yok, set var |
| **Learned position embedding** (BERT) | Slot 0..N için tablo | Slot kimliği keyfi; gezegen 0 her oyunda farklı yerde |
| **Relative position bias** (T5, ALiBi) | Çiftlere göre öğrenilmiş bias | Sıralı pozisyon temelli; bizde Öklid mesafesi var |
| **Raw (x,y) feature olarak ver** | Token feature'ına ekle | Mesafeyi Q/K'dan öğrenmek zor |

### Bizim çözüm

Her token *çifti* için 5 sayı:

```python
edge_bias[i, j] = (
    dist(i, j) / 100,         # mesafe
    (x_j - x_i) / 100,        # işaretli x farkı
    (y_j - y_i) / 100,        # işaretli y farkı
    sin(phi_j - phi_i),       # güneş etrafında açı farkı
    cos(phi_j - phi_i),       # güneş etrafında açı farkı
)
```

Burada `phi_t = atan2(y_t - sun_y, x_t - sun_x)`. İlk 3 kanal **kartezyen** ilişkiyi taşır (mesafe + işaretli yön). Son 2 kanal **güneş merkezli polar** ilişkiyi taşır.

Ayrım şöyle:

- **Token feature** = "ben kimim" (sahip, gemi, polar konum)
- **Edge feature** = "sen ve ben arasındaki ilişki" (mesafe, kartezyen yön, orbital faz farkı)

Bu ayrım, **graph transformer / Graphormer / GAT** literatüründe `edge attention bias` olarak bilinir.

### Neden hem `(dx, dy)` hem `(sin Δφ, cos Δφ)`?

Bu iki temsil aynı ham bilgiden türetilir ama farklı **invaryanslara** sahiptir, model her birinden farklı şey öğrenir:

| Özellik | `(dx, dy)` | `(sin Δφ, cos Δφ)` |
| --- | --- | --- |
| Boyut | 2 | 2 (cyclic, sin/cos) |
| Uzay tipi | Kartezyen, mutlak | Güneş merkezli polar, göreli |
| Board rotasyonuna ne olur? | Dönüyor (her turn değişir) | Değişmiyor (sabit) |
| Fleet/comet için anlamlı mı? | Evet, doğrudan | Kısmen (fleet orbit'te değil) |
| Sun avoidance için faydası | Doğrudan vektör | Açı bilgisi |

**Kritik gözlem:** Orbit Wars'ta tüm gezegenler **aynı** açısal hız `ω` ile döner. Yani turn `t`'de `phi_i(t) = phi_i(0) + ω·t`. Bu durumda iki gezegen arasında:

$$\Delta\varphi(i, j) = \varphi_j - \varphi_i = \text{sabit}$$

Bu, planet-planet çiftleri için **zamanla değişmeyen** bir feature anlamına gelir. Model "bu iki gezegen hep şu açı ilişkisinde" bilgisini sabit görür → her step'te spatial layout'u yeniden öğrenmek zorunda kalmaz.

`(dx, dy)` aksine her turn döndüğü için bu kalıcılığı yakalayamaz. Her ikisini birlikte vermek modele:
- Anlık geometrik ilişki (dx, dy) — fleet/sun pathfinding için
- Kalıcı orbital ilişki (sin/cos Δφ) — gezegen seçimi ve uzun vadeli planlama için

iki farklı zaman ölçeğinde kullanışlı sinyal sağlar.

**Neden tek skaler `Δφ` değil?** Açı cyclic'tir: `+π` ve `-π` aynı yönü gösterir ama sayısal olarak çok uzaktırlar. Tek skaler verirsek model bu süreksizliği `Linear` katmanıyla çözemez. `(sin, cos)` ise unit circle üstünde sürekli — `Linear(2, H)` sorunsuz çalışır.

### `edge_proj` ne yapıyor?

`Linear(5, n_heads)`. Yani 5-boyutlu edge feature'ını **head sayısına** projekte eder. Neden head başına 1 skaler?

**Fikir:** her head kendi uzamsal yorumunu öğrensin.

Öğrenildiğinde `W_edge ∈ R^{5×H}` matrisinin her sütunu bir head'in "geometrik filtresi" olur. Örneğin:

- **Head 1:** `w_1 ≈ [-3, 0, 0, 0, 0]` → "Yakın gezegenler önemli" (dist arttıkça bias düşer)
- **Head 2:** `w_2 ≈ [0, +2, 0, 0, 0]` → "Sağdaki (pozitif dx) gezegenleri terfi ettir"
- **Head 3:** `w_3 ≈ [0, 0, 0, +1, +1]` → "Orbital olarak benzer fazda olanlara bak" (Δφ ≈ 0)
- **Head 4:** `w_4 ≈ [0, 0, 0, 0, -1]` → "Karşı tarafta olanlara odaklan" (cos Δφ < 0 → Δφ ≈ π)
- **Head 5:** `w_5 ≈ [-1, 0, +1, 0, 0]` → "Hem yakın hem yukarıda olanlar"

Aynı `edge_bias` matrisi, farklı head'lerde farklı **uzamsal odaklar** yaratır. Önemlisi: bazı head'ler sadece kartezyen kanalları, bazıları sadece açısal kanalları kullanmayı öğrenebilir — bu seçim tamamen veriyle şekillenir.

### Softmax öncesi ekleme neden önemli?

```
softmax(Q·K/√d + bias)
```

Softmax expansiyonuyla:

$$\text{attn}(j) = \frac{e^{(QK)_{ij}/\sqrt{d}} \cdot e^{b_{ij}}}{\sum_k e^{(QK)_{ik}/\sqrt{d}} \cdot e^{b_{ik}}}$$

Yani bias **çarpan** gibi davranır. Pratik etkisi:

- `edge_proj` öğrenebilir ki `dist → -3·dist`. 100 birim uzaktaki gezegenin skoruna `-3` eklenir.
- `exp(-3) ≈ 0.05` → o key 20× daha az attention alır.
- Mesafe tabanlı **attention damping** otomatik olarak ortaya çıkar.

### Neden Q/K'ya bırakmadık?

Eğer `(x, y)`'yi token feature olarak verseydik:

$$QK_{ij} = Q(\text{plan}_i + x_i, y_i) \cdot K(\text{plan}_j + x_j, y_j)$$

Bu skor mesafenin doğrudan fonksiyonu değil; bilineer forma sıkışmış. Model `||x_i − x_j||²` türü non-lineer yapıyı sıfırdan öğrenmek zorunda — daha çok veri, daha yavaş yakınsama.

Edge bias ile mesafeyi ve açı farkını **servis ediyoruz**, model sadece `Linear(5, H)`'in ağırlıklarını optimize ediyor. Çok daha veri-verimli.

### Diyagonalin sıfırlanması

`edge_bias[i, i, :] = 0` zorunlu olarak ayarlanır. Aksi halde `cos(Δφ = 0) = 1` diyagonal üstünde sabit bir bias bırakır ve self-attention'ı yapay olarak yukarı çeker. Bu küçük detay, attention'ın içerik tabanlı `Q·K` bileşenini özgür bırakır.

### Maliyeti

- Memory: `(B, T, T, edge_dim)` edge tensoru + `(B, H, T, T)` projekte bias.
- Bizde `T = 65`, `edge_dim = 5`, `H = 4` → ihmal edilebilir.
- Ama `T = 4096` gibi uzun LLM dizilerinde `T² × H` patlar — bu yüzden NLP'de edge bias yaygın değil, ALiBi gibi kompakt alternatifler tercih edilir.

### Genişletme fikirleri (TODO)

Şu an `edge_dim = 5` (kartezyen + açısal). Eklenebilecek prior'lar:

- `eta_at_ref_speed = dist / fleet_speed(ref_ships)` → zaman cinsinden mesafe
- `bearing_alignment`: bir fleet'in gidiş açısı ile bearing arasındaki fark — "j fleet'i i'ye yaklaşıyor mu?"
- `same_owner`: aynı sahibe sahip gezegenleri işaretleyen flag
- `is_blocked`: aralarında başka gezegen olup olmadığını gösteren binary (path collision sinyali)

Her ek feature, `edge_proj`'e yeni bir satır ekler; head'ler daha zengin filtreler öğrenir.

### Akademik referanslar

- **Graphormer** (Ying et al., NeurIPS 2021): `B_{ij} = \text{Linear}(\text{shortest\_path\_features})` head başına. Birebir aynı pattern.
- **GAT — Graph Attention Network** (Veličković et al., 2018): edge feature concat edilir, MLP'den geçirilir.
- **ALiBi** (Press et al., 2022): edge bias'ın LLM versiyonu — relative position'ı `−m·|i−j|` ile baskılar.

### Tek cümlelik özet

> Token feature "ben kimim"; edge bias "sen ve ben arasında ne var". Mesafe ve kartezyen yön anlık geometriyi, güneşe göre `(sin Δφ, cos Δφ)` ise board rotasyonundan bağımsız orbital ilişkiyi attention'ın **additive** skor tarafına taşır; softmax bunu multiplicative damping'e dönüştürür. Her head `edge_proj`'un bir sütunu ile **kendi geometrik filtresini** öğrenir.

## 8. Padding Mask: Set Boyutu Değişken, Tensor Boyutu Sabit

Attention içinde padding tokenlar key olarak maskelenir:

```python
mask = key_padding_mask[:, None, None, :]
scores = scores.masked_fill(mask, float('-inf'))
```

Yani gerçek tokenlar padding tokenlara attention veremez. Padding query satırları teknik olarak var kalır; ama onların çıktısı downstream aksiyonlarda zaten maskelendiği için semantik bir karşılık taşımaz.

`torch.nan_to_num` güvenlik için kullanılır. Eğer bir attention satırı tamamen `-inf` olursa softmax `NaN` üretebilir; kod bunu sıfıra çeker.

## 9. TransformerBlock: Pre-Norm Residual Yapı

Her blok şu akışı izler:

$$X' = X + \mathrm{Attention}(\mathrm{LN}(X))$$

$$Y = X' + \mathrm{FFN}(\mathrm{LN}(X'))$$

Bu `pre-norm` tasarımdır. LayerNorm attention/FFN öncesinde uygulanır. Derin Transformer eğitiminde pre-norm genellikle daha stabil gradyan akışı verir.

FFN kısmı iki lineer katman ve GELU aktivasyonundan oluşur:

```python
Linear(d_model, 4*d_model) -> GELU -> Linear(4*d_model, d_model)
```

## 10. Actor: Aksiyonu Parçalayarak Modellemek

Orbit Wars aksiyonu tek bir kategorik seçim değildir. Bir ham env aksiyonu kabaca şudur:

```text
[from_planet_id, angle, ships]
```

Model bunu slot tabanlı üç karara ayırır:

$$p(a|s) = p(src|s)\;p(tgt|src,s)\;p(ship\_bin|src,s)$$

Bu faktörizasyon kombinatorik aksiyon uzayını küçültür. Doğrudan tüm `(src, tgt, ship)` kombinasyonlarına ayrı logit üretmek yerine:

- Kaynak için `(P_max + 1)` logit üretilir. Son slot NO-OP'tur.
- Hedef için her kaynak gezegenden her hedef gezegene pointer skoru üretilir.
- Gemi oranı için her kaynak gezegen başına `n_ship_bins` logit üretilir.

Varsayılan ship bin anlamları `decode_action` tarafında şuna karşılık gelir:

```python
(0.1, 0.25, 0.5, 0.75, 1.0)
```

Mevcut PPO policy turn başına tek composite action üretir. Yeni `incoming_self` ve `incoming_enemy_0/1/2` planet feature'ları, sonraki adımda sequential multi-action tasarımını kolaylaştırır: aynı turn içinde bir launch üretildikten sonra temporary observation'da source ships azaltılıp target `incoming_self_*` kanalları artırılırsa, sonraki source kararı önceki launch'ı state içinde görebilir.

In [6]:
with torch.no_grad():
    out = net(**batch)

print('src_logits :', tuple(out.src_logits.shape), 'P_max + NO-OP')
print('tgt_logits :', tuple(out.tgt_logits.shape), 'source x target')
print('ship_logits:', tuple(out.ship_logits.shape), 'source x ship_bin')
print('value      :', tuple(out.value.shape))
print('tokens     :', tuple(out.tokens.shape))

src_logits : (1, 9) P_max + NO-OP
tgt_logits : (1, 8, 8) source x target
ship_logits: (1, 8, 5) source x ship_bin
value      : (1,)
tokens     : (1, 13, 32)


## 11. Source Head ve NO-OP

Kaynak seçimi gezegen tokenları üzerinden yapılır:

```python
src_raw = self.src_head(planet_tok).squeeze(-1)  # (B, P)
src_raw = src_raw.masked_fill(~source_mask, very_neg)
src_logits = torch.cat([src_raw, noop], dim=1)  # (B, P+1)
```

`source_mask` sadece bize ait ve yeterli gemisi olan gezegenleri açık bırakır. Böylece model illegal kaynaklardan filo çıkarma olasılığı öğrenmez; bu olasılık en baştan sıfırlanır.

NO-OP ayrı, öğrenilebilir bir logittir. Bu önemli çünkü bazı durumlarda hiçbir şey yapmamak en iyi aksiyon olabilir.

In [7]:
src_probs = torch.softmax(out.src_logits, dim=-1)

print('source_mask       :', batch['source_mask'][0].int().tolist())
print('source prob       :', src_probs[0].round(decimals=4).tolist())
print('son eleman NO-OP  :', src_probs[0, -1].item())
print('invalid source sum:', src_probs[0, 2:8].sum().item())

source_mask       : [1, 1, 0, 0, 0, 0, 0, 0]
source prob       : [0.423799991607666, 0.3978999853134155, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.17829999327659607]
son eleman NO-OP  : 0.17825359106063843
invalid source sum: 0.0


## 12. Target Head: Pointer Attention

Hedef seçimi, kaynak gezegen tokenından hedef gezegen tokenlarına doğru bir pointer skoru gibi hesaplanır:

$$score(src,tgt) = \frac{q_{src} \cdot k_{tgt}}{\sqrt{d_{model}}}$$

Kod:

```python
q = self.tgt_query(planet_tok)
k = self.tgt_key(planet_tok)
tgt_logits = torch.matmul(q, k.transpose(-2, -1)) / sqrt(d_model)
```

Sonra iki maske uygulanır:

- `target == source` yasaktır.
- Padding gezegenler hedef olamaz.

Bu nedenle diagonal ve padding target kolonları `very_neg` yapılır.

In [8]:
tgt_probs = torch.softmax(out.tgt_logits, dim=-1)

print('source 0 için target olasılıkları:')
print(tgt_probs[0, 0].round(decimals=4).tolist())
print('self target prob  :', tgt_probs[0, 0, 0].item())
print('padding target sum:', tgt_probs[0, 0, 4:].sum().item())

source 0 için target olasılıkları:
[0.0, 0.35690000653266907, 0.32100000977516174, 0.3221000134944916, 0.0, 0.0, 0.0, 0.0]
self target prob  : 0.0
padding target sum: 0.0


## 13. Ship Head: Sürekli Sayıyı Ayrık Bin'e Çevirmek

Geminin tam sayısını doğrudan regression ile tahmin etmek yerine model ayrık oran bin'i seçer:

```python
ship_logits = self.ship_head(planet_tok)  # (B, P, n_ship_bins)
```

Sonradan `decode_action`, seçilen bin'i kaynak gezegenin mevcut gemi sayısıyla çarpar. Örneğin bin `0.5` ise mevcut gemilerin yaklaşık yarısı gönderilir.

Bu seçim RL için pratik bir avantaj sağlar: policy dağılımı kategorik kalır, log-prob ve entropy hesapları kolaylaşır.

## 14. Critic: Global Token Üzerinden Value

Transformer bloklarından sonra son token global tokendır:

```python
global_tok = x[:, -1]
value = self.value_head(global_tok).squeeze(-1)
```

Global token başlangıçta global feature'lardan gelir; self-attention katmanlarından geçerken tüm gezegen ve filo tokenlarından bilgi toplayabilir. Bu yüzden value head için doğal bir havuzlanmış temsil gibi davranır.

PPO/A2C gibi actor-critic algoritmalarında bu `V(s)` tahminidir.

## 15. `act`: Dağılımdan Aksiyon Örneklemek

`act` metodu `PolicyOutput` içindeki logitlerden kategorik dağılımlar kurar:

1. `src ~ Categorical(src_logits)`
2. Seçilen `src` ile ilgili target satırı alınır.
3. `tgt ~ Categorical(tgt_logits[src])`
4. Seçilen `src` ile ilgili ship-bin satırı alınır.
5. `ship_bin ~ Categorical(ship_logits[src])`

NO-OP seçildiyse target ve ship teknik olarak yine shape tutarlılığı için örneklenir; ama log-prob hesabında target/ship katkısı sıfırlanır:

$$\log p(a|s) = \log p(src|s) + \mathbb{1}_{src \ne NOOP}\left[\log p(tgt|src,s) + \log p(ship|src,s)\right]$$

In [9]:
action = net.act(out, deterministic=False)
for k, v in action.items():
    print(f'{k:8s}:', v.tolist())

det_action = net.act(out, deterministic=True)
print('\nDeterministik aksiyon:')
for k, v in det_action.items():
    print(f'{k:8s}:', v.tolist())

src     : [1]
tgt     : [2]
ship_bin: [3]
is_noop : [False]
log_prob: [-2.7768025398254395]

Deterministik aksiyon:
src     : [0]
tgt     : [1]
ship_bin: [3]
is_noop : [False]
log_prob: [-2.57684063911438]


## 16. Slot'tan Env Aksiyonuna Dönmek

Model gezegen id'si değil slot index'i seçer. `decode_action` slotları tekrar env id'lerine çevirir:

```python
src_id = int(obs.planet_ids[src_slot])
tgt_id = int(obs.planet_ids[tgt_slot])
```

Ardından `navigation.compute_launch` ile gerçek açı hesaplanır. Böylece policy ağı doğrudan fizik/geometri çözmek zorunda kalmaz; stratejik seçimi yapar, deterministik navigation katmanı bunu env aksiyonuna dönüştürür.

Bu ayrım mimari olarak temizdir:

- Model: hangi kaynaktan, hangi hedefe, ne kadar gemi?
- Navigation: hangi açıyla fırlatılırsa hedefe gider?

## 17. Neden Bu Mimari İyi Bir Uyum?

| Orbit Wars ihtiyacı | Modeldeki karşılık |
| --- | --- |
| Değişken sayıda nesne | Padding + mask |
| Sıra bağımsız gezegen/filo listeleri | Self-attention tabanlı set encoder |
| Mesafe ve yön çok önemli | Edge bias ile attention skorlarına geometri |
| Illegal aksiyonlar sık | Source/target mask |
| `from, target, ships` kombinatorik uzayı | Faktörize actor head |
| RL için value tahmini | Global token critic |

Özellikle edge bias tasarımı oyuna iyi oturur. Saf Transformer koordinatlardan mesafeyi öğrenebilir; ama burada mesafe ve yön zaten oyun mekaniğinin merkezinde olduğu için bu bilgiyi attention skoruna doğrudan vermek daha veri verimli bir başlangıç sağlar.

## 18. Eğitimde Kullanım Notları

Bu model PPO/A2C tarzı actor-critic eğitim için doğrudan uygundur:

- `out.value`: critic tahmini `V(s)`
- `action['log_prob']`: policy loss için log olasılık
- `src_logits`, `tgt_logits`, `ship_logits`: entropy bonus için kullanılabilir

Dikkat edilmesi gereken noktalar:

- Target/ship entropy NO-OP satırlarında hesaba katılırsa yapay entropy oluşabilir; NO-OP için bu parçaları maskeli hesaplamak daha doğru olur.
- `source_mask` tamamen boşsa bile NO-OP her zaman açık kaldığı için policy geçerli dağılım üretir.
- `very_neg = torch.finfo(dtype).min` finite olduğu için testlerde `isfinite` kontrolü geçer; softmax sonrasında pratik olarak sıfır olasılık verir.
- `edge_proj` her head için farklı bias öğrenir; bazı head'ler yakın hedeflere, bazıları tehdit yönüne veya global token ilişkisine uzmanlaşabilir.

## 19. Kısa Kod Haritası

| `model.py` parçası | Görev |
| --- | --- |
| `BiasedMultiHeadAttention` | QKV attention + edge bias + padding mask |
| `TransformerBlock` | Pre-norm attention ve FFN residual bloğu |
| `PolicyOutput` | Actor/critic çıktılarını isimli taşıyıcıda toplar |
| `OrbitWarsPolicy.__init__` | Projeksiyonlar, encoder blokları ve head'leri kurar |
| `_build_tokens` | Planet/fleet/global feature'larını token dizisine çevirir |
| `_build_attn_bias` | `(B,T,T,E)` edge feature'ı `(B,H,T,T)` bias'a çevirir |
| `_build_padding_mask` | Gerçek token / padding ayrımını attention maskesine çevirir |
| `forward` | Encoder, actor head'leri ve value head'i çalıştırır |
| `act` | Logitlerden aksiyon örnekler ve log-prob döndürür |
| `pack_observations` | Observation listesini batch tensor dict'e çevirir |